In [2]:
data = """
SP	544'663
FDP	312'554
Grüne	177'440
GLP	195'762
SVP	222'020
AL	100'054
Mitte	99'068
EVP	27'954
Diverse (ZVP/EDU/PdAZ/Freie Liste)	29952
"""

In [7]:
results = dict(map(lambda line: line.replace("'", "").split("\t"), data.strip().split("\n")))

In [10]:
import polars as pl

In [18]:
df = pl.DataFrame({'partei': results.keys(), 'stimmen': results.values()}).with_columns(pl.col('stimmen').cast(pl.Int64))

In [21]:

missed = ['EVP', 'Diverse (ZVP/EDU/PdAZ/Freie Liste)']

df

partei,stimmen
str,i64
"""SP""",544663
"""FDP""",312554
"""Grüne""",177440
"""GLP""",195762
"""SVP""",222020
"""AL""",100054
"""Mitte""",99068
"""EVP""",27954
"""Diverse (ZVP/EDU/PdAZ/Freie Li…",29952


In [ ]:
total_seats = 125

In [ ]:
# Oberzuteilung (Sainte-Laguë/Webster) for 125 seats, excluding parties in `missed`
seats_to_allocate = total_seats  # already defined as 125

eligible_df = df.filter(~pl.col("partei").is_in(missed))
votes = dict(zip(eligible_df["partei"].to_list(), eligible_df["stimmen"].to_list()))

ober_seat_counts = {party: 0 for party in votes}

for _ in range(seats_to_allocate):
    winner = max(votes, key=lambda p: votes[p] / (2 * ober_seat_counts[p] + 1))
    ober_seat_counts[winner] += 1

# add excluded parties with 0 seats
all_parties = df["partei"].to_list()
ober_seat_counts_full = {p: ober_seat_counts.get(p, 0) for p in all_parties}

ober_seat_df = pl.DataFrame({
    "partei": all_parties,
    "sitze": [ober_seat_counts_full[p] for p in all_parties],
})

ober_df = (
    df.join(ober_seat_df, on="partei", how="left")
      .with_columns(
          pl.when(pl.col("sitze") > 0)
            .then((pl.col("stimmen") / pl.col("sitze")).round(2))
            .otherwise(None)
            .alias("waehlerzahl")
      )
      .sort("sitze", descending=True)
)

ober_df